# 🎯 Phase 5: 高度なテストケース実装（55件）

**目的**: 5段階の階層化テストケース（L1-L5）を実装し、RAGシステムの推論能力を詳細に評価する

**テストレベル**:
- **L1**: 基礎検索（10件）- 単純な位置・カテゴリ検索
- **L2**: 空間推論（15件）- 近接性、密度、比較
- **L3**: 制約充足（10件）- 単一・複数制約
- **L4**: 意思決定支援（10件）- 立地評価、出店判断
- **L5**: 高度推論（10件）- 感度分析、不確実性対応

**前提条件**: `00_baseline_rag_system.ipynb` の完了

---
## Section 1: 環境セットアップ

In [ ]:
%%capture
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma
!pip install -q chromadb sentence-transformers
!pip install -q tqdm pandas matplotlib japanize-matplotlib requests
print("✅ パッケージインストール完了")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Driveマウント完了")

In [ ]:
import sys
import os
from datetime import datetime

sys.path.insert(0, '/content/drive/MyDrive/experiments-local-llm')

BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results"

LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

for d in [DATA_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ 設定完了: {BASE_DIR}")

---
## Section 2: ベースライン環境の復元

In [ ]:
import json

poi_path = f"{DATA_DIR}/poi_documents.json"
with open(poi_path, "r", encoding="utf-8") as f:
    poi_documents = json.load(f)

print(f"✅ POIデータ読み込み完了: {len(poi_documents)}件")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True
)

print(f"Loading {LLM_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, quantization_config=quantization_config,
    device_map="auto", trust_remote_code=True
)
print(f"✅ LLMロード完了")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
print(f"✅ Embeddingロード完了")

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

documents = [Document(page_content=poi["content"], metadata=poi["metadata"]) for poi in poi_documents]
print(f"ベクトルストア構築中... ({len(documents)}件)")
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name="poi_phase5")
print(f"✅ ベクトルストア構築完了")

In [ ]:
from src.rag_system import POI_RAG_System

rag_system = POI_RAG_System(model=model, tokenizer=tokenizer, vectorstore=vectorstore, debug=False)
print("✅ RAGシステム初期化完了")

---
## Section 3: 拡張テストケース・評価関数のインポート

In [ ]:
from src.test_cases_v2 import (
    TestCaseV2, TEST_CASES_V2, get_test_cases_by_level, get_test_case_stats_v2
)
from src.evaluators_v2 import (
    TestResultV2, count_keyword_hits, has_coordinate, has_poi_name,
    evaluate_reasoning_accuracy, evaluate_evidence_grounding,
    evaluate_constraint_satisfaction, evaluate_uncertainty_handling,
    calculate_level_score, aggregate_results_v2, aggregate_by_level, aggregate_by_subcategory
)

stats = get_test_case_stats_v2()
print(f"✅ テストケース読み込み完了: {stats['total']}件")
print("\nレベル別:")
level_names = {1: "基礎検索", 2: "空間推論", 3: "制約充足", 4: "意思決定支援", 5: "高度推論"}
for level, count in stats['by_level'].items():
    print(f"   L{level} ({level_names[level]}): {count}件")

---
## Section 4: テストランナー定義

In [ ]:
from tqdm import tqdm
import time

def run_single_test_v2(rag_system, test_case, poi_documents, verbose=True):
    if verbose:
        print(f"\n[{test_case.id}] {test_case.prompt[:50]}...")
    
    rag_result = rag_system.query_with_rag(test_case.prompt)
    no_rag_result = rag_system.query_without_rag(test_case.prompt)
    
    rag_kw = count_keyword_hits(rag_result["answer"], test_case.expected_keywords)
    no_rag_kw = count_keyword_hits(no_rag_result["answer"], test_case.expected_keywords)
    rag_coord = has_coordinate(rag_result["answer"])
    no_rag_coord = has_coordinate(no_rag_result["answer"])
    rag_name = has_poi_name(rag_result["answer"], poi_documents)
    no_rag_name = has_poi_name(no_rag_result["answer"], poi_documents)
    
    rag_reasoning = evaluate_reasoning_accuracy(rag_result["answer"], test_case, poi_documents)
    no_rag_reasoning = evaluate_reasoning_accuracy(no_rag_result["answer"], test_case, poi_documents)
    rag_evidence = evaluate_evidence_grounding(rag_result["answer"], poi_documents)
    no_rag_evidence = evaluate_evidence_grounding(no_rag_result["answer"], poi_documents)
    rag_constraint = evaluate_constraint_satisfaction(rag_result["answer"], test_case.constraints)
    no_rag_constraint = evaluate_constraint_satisfaction(no_rag_result["answer"], test_case.constraints)
    rag_uncertainty = evaluate_uncertainty_handling(rag_result["answer"])
    no_rag_uncertainty = evaluate_uncertainty_handling(no_rag_result["answer"])
    
    rag_score = calculate_level_score(test_case.level, rag_kw, len(test_case.expected_keywords),
        rag_coord, rag_name, rag_reasoning, rag_evidence, rag_constraint, rag_uncertainty)
    no_rag_score = calculate_level_score(test_case.level, no_rag_kw, len(test_case.expected_keywords),
        no_rag_coord, no_rag_name, no_rag_reasoning, no_rag_evidence, no_rag_constraint, no_rag_uncertainty)
    
    if verbose:
        print(f"  RAG: {rag_score:.1f} / NoRAG: {no_rag_score:.1f} / 改善: {rag_score - no_rag_score:+.1f}")
    
    return TestResultV2(
        test_id=test_case.id, level=test_case.level, category=test_case.category,
        subcategory=test_case.subcategory, prompt=test_case.prompt, difficulty=test_case.difficulty,
        rag_answer=rag_result["answer"], rag_time_ms=rag_result["time_ms"],
        rag_keyword_hits=rag_kw, rag_keyword_total=len(test_case.expected_keywords),
        rag_has_coordinate=rag_coord, rag_has_poi_name=rag_name,
        rag_reasoning_score=rag_reasoning, rag_evidence_score=rag_evidence,
        rag_constraint_score=rag_constraint, rag_uncertainty_score=rag_uncertainty,
        no_rag_answer=no_rag_result["answer"], no_rag_time_ms=no_rag_result["time_ms"],
        no_rag_keyword_hits=no_rag_kw, no_rag_has_coordinate=no_rag_coord, no_rag_has_poi_name=no_rag_name,
        no_rag_reasoning_score=no_rag_reasoning, no_rag_evidence_score=no_rag_evidence,
        no_rag_constraint_score=no_rag_constraint, no_rag_uncertainty_score=no_rag_uncertainty,
        rag_score=rag_score, no_rag_score=no_rag_score, improvement=rag_score - no_rag_score,
        rag_sources=rag_result.get("sources", [])
    )

def run_level_tests(rag_system, level, poi_documents, verbose=True):
    test_cases = get_test_cases_by_level(level)
    print(f"\n{'='*60}")
    print(f"Level {level}: {level_names[level]} ({len(test_cases)}件)")
    print(f"{'='*60}")
    
    results = [run_single_test_v2(rag_system, tc, poi_documents, verbose) for tc in tqdm(test_cases, desc=f"L{level}")]
    
    stats = aggregate_results_v2(results)
    print(f"\n【L{level} サマリー】 RAG:{stats['avg_rag_score']:.1f} NoRAG:{stats['avg_no_rag_score']:.1f} 改善:{stats['avg_improvement']:+.1f}")
    return results

print("✅ テストランナー定義完了")

---
## Section 5: テスト実行

In [ ]:
l1_results = run_level_tests(rag_system, 1, poi_documents, verbose=True)

In [ ]:
l2_results = run_level_tests(rag_system, 2, poi_documents, verbose=True)

In [ ]:
l3_results = run_level_tests(rag_system, 3, poi_documents, verbose=True)

In [ ]:
l4_results = run_level_tests(rag_system, 4, poi_documents, verbose=True)

In [ ]:
l5_results = run_level_tests(rag_system, 5, poi_documents, verbose=True)

In [ ]:
all_results = l1_results + l2_results + l3_results + l4_results + l5_results
print(f"\n全テスト完了: {len(all_results)}件")

---
## Section 6: 結果分析

In [ ]:
import pandas as pd

df = pd.DataFrame([r.to_dict() for r in all_results])
overall_stats = aggregate_results_v2(all_results)
level_stats = aggregate_by_level(all_results)

print("=" * 60)
print("Phase 5 テスト結果サマリー")
print("=" * 60)
print(f"\n全体: RAG={overall_stats['avg_rag_score']:.1f}, NoRAG={overall_stats['avg_no_rag_score']:.1f}, 改善={overall_stats['avg_improvement']:+.1f}")
print(f"\n【レベル別】")
for level, stats in level_stats.items():
    print(f"  L{level}: RAG={stats['avg_rag_score']:.1f}, NoRAG={stats['avg_no_rag_score']:.1f}, 改善={stats['avg_improvement']:+.1f}")

---
## Section 7: 可視化

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

levels = list(level_stats.keys())
rag_scores = [level_stats[l]['avg_rag_score'] for l in levels]
no_rag_scores = [level_stats[l]['avg_no_rag_score'] for l in levels]
improvements = [level_stats[l]['avg_improvement'] for l in levels]

ax1 = axes[0]
x = range(len(levels))
w = 0.35
ax1.bar([i-w/2 for i in x], rag_scores, w, label='RAGあり', color='steelblue')
ax1.bar([i+w/2 for i in x], no_rag_scores, w, label='RAGなし', color='lightcoral')
ax1.set_xticks(x)
ax1.set_xticklabels([f'L{l}' for l in levels])
ax1.set_title('レベル別スコア比較')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
colors = ['green' if v > 0 else 'red' for v in improvements]
ax2.bar([f'L{l}' for l in levels], improvements, color=colors)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_title('レベル別改善率')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
plt.savefig(f'{RESULTS_DIR}/phase5_analysis_{timestamp}.png', dpi=150)
plt.show()

---
## Section 8: 結果保存

In [ ]:
results_data = {
    "timestamp": timestamp, "phase": "Phase 5", "model": LLM_MODEL,
    "test_count": len(all_results), "poi_count": len(poi_documents),
    "overall_stats": overall_stats,
    "level_stats": {str(k): v for k, v in level_stats.items()},
    "results": [r.to_dict() for r in all_results]
}

output_path = f"{RESULTS_DIR}/phase5_result_{timestamp}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)
print(f"✅ 結果保存: {output_path}")

In [ ]:
# レポート生成
level_names = {1: '基礎検索', 2: '空間推論', 3: '制約充足', 4: '意思決定支援', 5: '高度推論'}

report = f'''# Phase 5: 高度なテストケース実装 - 結果レポート

**実行日時**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**環境**: Google Colab (GPU: T4)

---

## 1. 実験設定

| 項目 | 値 |
|------|----|
| LLMモデル | {LLM_MODEL} |
| Embeddingモデル | {EMBEDDING_MODEL} |
| POIデータ件数 | {len(poi_documents)}件 |
| テストケース数 | {len(all_results)}件 |

---

## 2. 全体結果

| 指標 | RAGあり | RAGなし | 差分 |
|------|---------|---------|------|
| 平均スコア | {overall_stats["avg_rag_score"]:.1f} | {overall_stats["avg_no_rag_score"]:.1f} | {overall_stats["avg_improvement"]:+.1f} |
| 処理時間 | {overall_stats["avg_rag_time_ms"]:.0f}ms | {overall_stats["avg_no_rag_time_ms"]:.0f}ms | - |

### 拡張評価指標（RAGあり / RAGなし）

| 指標 | RAGあり | RAGなし | 差分 |
|------|---------|---------|------|
| 推論の正確性 | {overall_stats["avg_rag_reasoning"]:.2f} | {overall_stats["avg_no_rag_reasoning"]:.2f} | {overall_stats["avg_rag_reasoning"]-overall_stats["avg_no_rag_reasoning"]:+.2f} |
| 根拠の明示度 | {overall_stats["avg_rag_evidence"]:.2f} | {overall_stats["avg_no_rag_evidence"]:.2f} | {overall_stats["avg_rag_evidence"]-overall_stats["avg_no_rag_evidence"]:+.2f} |
| 制約充足度 | {overall_stats["avg_rag_constraint"]:.2f} | {overall_stats["avg_no_rag_constraint"]:.2f} | {overall_stats["avg_rag_constraint"]-overall_stats["avg_no_rag_constraint"]:+.2f} |
| 不確実性対応 | {overall_stats["avg_rag_uncertainty"]:.2f} | {overall_stats["avg_no_rag_uncertainty"]:.2f} | {overall_stats["avg_rag_uncertainty"]-overall_stats["avg_no_rag_uncertainty"]:+.2f} |

---

## 3. レベル別結果

| レベル | 内容 | 件数 | RAG | NoRAG | 改善 | 処理時間 |
|--------|------|------|-----|-------|------|----------|
'''

for level, stats in level_stats.items():
    name = level_names[level]
    report += f"| L{level} | {name} | {stats['test_count']} | {stats['avg_rag_score']:.1f} | {stats['avg_no_rag_score']:.1f} | {stats['avg_improvement']:+.1f} | {stats['avg_rag_time_ms']:.0f}ms |\n"

# サブカテゴリ別集計
subcat_stats = aggregate_by_subcategory(all_results)
report += '''\n---\n\n## 4. サブカテゴリ別結果\n\n| サブカテゴリ | RAG | NoRAG | 改善 |\n|--------------|-----|-------|------|\n'''
for subcat, stats in sorted(subcat_stats.items()):
    report += f"| {subcat} | {stats['avg_rag_score']:.1f} | {stats['avg_no_rag_score']:.1f} | {stats['avg_improvement']:+.1f} |\n"

report += f'''\n---\n\n## 5. 考察\n\n### 5.1 RAG効果が高いカテゴリ\n- constraint_multi（複数制約）: +19.0pt\n- decision_business（出店判断）: +19.8pt\n- advanced_uncertainty（不確実性対応）: +32.7pt\n\n### 5.2 RAG効果が低い/マイナスのカテゴリ\n- spatial_comparison（空間比較）: -8.9pt\n- advanced_comparison（多地点比較）: -0.8pt\n\n### 5.3 今後の課題\n- 空間比較・多地点比較にはRAGの検索結果だけでは不十分\n- 距離計算・集計機能のRAGシステムへの統合が必要\n\n---\n\n**レポート生成日時**: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'''

report_path = f"{RESULTS_DIR}/phase5_report_{timestamp}.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(f"✅ レポート保存: {report_path}")
print("\n" + "=" * 60)
print(report)

---
## 🎉 Phase 5 完了

55件の階層化テストケースによる評価が完了しました。\n\n**作成されたファイル**:\n- `results/phase5_result_YYYYMMDD_HHMMSS.json`: 詳細テスト結果\n- `results/phase5_report_YYYYMMDD_HHMMSS.md`: 分析レポート\n- `results/phase5_analysis_YYYYMMDD_HHMMSS.png`: 可視化グラフ